# Assignment 2 — Deep Learning on Cityscapes (Google Colab Fast Runner)
### Fully automated end-to-end training and evaluation pipeline for Semantic Segmentation on Cityscapes using DeepLabV3 (ResNet-50).

> **📌 HƯỚNG DẪN THỰC HIỆN ĐƠN GIẢN:**
> 1. Điền toàn bộ thông tin tại **BẢNG CẤU HÌNH TỔNG HỢP** ở ngay Cell bên dưới:
>    - **Tài khoản Cityscapes**: `CITYSCAPES_USERNAME` và `CITYSCAPES_PASSWORD` (đăng ký miễn phí tại [cityscapes-dataset.com](https://www.cityscapes-dataset.com/register/)) để Colab tải trực tiếp từ web với tốc độ cao, **hoàn toàn không cần Google Drive**.
>    - **GitHub Personal Access Token** *(tùy chọn)*: Để tự động đẩy `log.csv` và `metrics_test.json` về GitHub sau mỗi mô hình.
>    - **Hyperparameters**: Epochs, Batch Size, Learning Rates, Seed...
> 2. Bấm menu **Runtime $\rightarrow$ Run all** (hoặc nhấn tổ hợp phím **Ctrl + F9**).
> 3. Hệ thống sẽ tự động thực hiện: Tải dataset từ web $\rightarrow$ Giải nén $\rightarrow$ Sinh mask trainIds $\rightarrow$ Tự tinh chỉnh GPU $\rightarrow$ Train 3 model $\rightarrow$ Đánh giá trên test split $\rightarrow$ Xuất bảng so sánh kết quả!

In [ ]:
# @title 🎛️ BẢNG CẤU HÌNH TỔNG HỢP / UNIFIED CONFIGURATION PANEL
# @markdown Hãy điền toàn bộ thông tin tại đây trước khi bấm Run All. Sau bước này, bạn không cần chỉnh sửa bất kỳ ô nào khác.

# @markdown ---
# @markdown ### 🌐 1. TẢI DATASET TRỰC TIẾP TỪ WEB (Không cần Google Drive)
# @markdown Đăng ký tài khoản miễn phí tại: https://www.cityscapes-dataset.com/register/
CITYSCAPES_USERNAME = ""  # @param {type:"string"}
CITYSCAPES_PASSWORD = ""  # @param {type:"string"}

# @markdown *(Tùy chọn) Nếu bạn có link trực tiếp tải file zip (mirror / HuggingFace / Direct URL) thì điền vào đây:*
CUSTOM_GTFINE_URL = ""  # @param {type:"string"}
CUSTOM_LEFTIMG_URL = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🐙 2. GITHUB AUTO-PUSH (Tùy chọn - Tự động đẩy kết quả lên GitHub)
# @markdown Personal Access Token tạo tại github.com/settings/tokens (quyền 'repo')
import os
try:
    from google.colab import userdata
    _token = userdata.get('GITHUB_TOKEN')
except Exception:
    _token = None

GITHUB_TOKEN = _token or ""  # @param {type:"string"}
GIT_USERNAME = "Cao Đức Thành"  # @param {type:"string"}
GIT_EMAIL = "ducthanh4225@gmail.com"  # @param {type:"string"}
REPO_OWNER = "phsuuuuuu"  # @param {type:"string"}
REPO_NAME = "CO3133_sem261_groupRTX5090"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}

# @markdown ---
# @markdown ### ⚙️ 3. HYPERPARAMETERS & HUẤN LUYỆN
EPOCHS = 10  # @param {type:"integer"}
RANDOM_SEED = 666  # @param {type:"integer"}
EVAL_EVERY = 1  # @param {type:"integer"}

# @markdown Chế độ Batch Size ('Auto' sẽ tự đo VRAM A100/L4/T4 để tối ưu tốc độ tối đa):
BATCH_SIZE_MODE = "Auto (Detect GPU)"  # @param ["Auto (Detect GPU)", "Custom"]
CUSTOM_BATCH_SIZE = 8  # @param {type:"integer"}
CUSTOM_NUM_WORKERS = 4  # @param {type:"integer"}
USE_AMP = True  # @param {type:"boolean"}

# @markdown Tốc độ học (Learning Rates):
HEAD_LR = 0.001  # @param {type:"number"}
BACKBONE_LR_PARTIAL = 0.0001  # @param {type:"number"}
BACKBONE_LR_FULL = 0.00001  # @param {type:"number"}

# @markdown Dry-run test nhanh (Đặt 0 để huấn luyện full dataset; đặt > 0 ví dụ 20 để test thử 20 batches):
MAX_TRAIN_BATCHES = 0  # @param {type:"integer"}
MAX_EVAL_BATCHES = 0  # @param {type:"integer"}

print("=" * 75)
print("🎛️  THÔNG TIN CẤU HÌNH ĐÃ THIẾT LẬP:")
print(f"   • Nguồn tải dataset      : {'Tài khoản Cityscapes Portal' if CITYSCAPES_USERNAME else ('Link tải trực tiếp' if CUSTOM_GTFINE_URL else '⚠️ CHƯA ĐIỀN')}")
print(f"   • GitHub Auto-Push       : {'BẬT (User: ' + GIT_USERNAME + ')' if GITHUB_TOKEN else 'TẮT (Sẽ tải file về máy ở ô cuối)'}")
print(f"   • Số Epochs mỗi mô hình  : {EPOCHS}")
print(f"   • Chế độ Batch Size      : {BATCH_SIZE_MODE} (Custom={CUSTOM_BATCH_SIZE})")
print(f"   • Head Learning Rate     : {HEAD_LR}")
print(f"   • Partial Backbone LR    : {BACKBONE_LR_PARTIAL}")
print(f"   • Full Backbone LR       : {BACKBONE_LR_FULL}")
print(f"   • Mixed Precision (AMP)  : {USE_AMP}")
print(f"   • Random Seed            : {RANDOM_SEED}")
print(f"   • Giới hạn Batches       : {'Toàn bộ Dataset' if MAX_TRAIN_BATCHES == 0 else f'{MAX_TRAIN_BATCHES} batches (Dry-run)'}")
print("=" * 75)

if not CITYSCAPES_USERNAME and not CUSTOM_GTFINE_URL:
    print("\n⚠️  CHÚ Ý: Bạn chưa nhập thông tin đăng nhập Cityscapes (hoặc link mirror)!")
    print("👉 Hãy điền CITYSCAPES_USERNAME và CITYSCAPES_PASSWORD ở form trên để tự động tải dữ liệu.")
    print("👉 Nếu chưa có tài khoản, hãy đăng ký nhanh tại: https://www.cityscapes-dataset.com/register/")

## Bước 1: Kiểm Tra Tài Nguyên Phần Cứng GPU

In [ ]:
# @title Kiểm Tra Thông Số GPU
!nvidia-smi

import torch
print(f"\nPyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name        : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## Bước 2: Clone / Cập Nhật Repository & Cài Đặt Gói Cần Thiết

In [ ]:
# @title Clone Repository và Cài Đặt Môi Trường
import os

REPO_URL = "https://github.com/phsuuuuuu/CO3133_sem261_groupRTX5090.git"
ROOT_DIR = "/content/CO3133_sem261_groupRTX5090"
ASSIGNMENT_DIR = f"{ROOT_DIR}/assignment2"

if not os.path.exists(ROOT_DIR):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} {ROOT_DIR}
else:
    print("Repository đã tồn tại. Đang đồng bộ commit mới nhất...")
    !git -C {ROOT_DIR} checkout -- assignment2/configs/*.yaml 2>/dev/null || true
    !git -C {ROOT_DIR} checkout main
    !git -C {ROOT_DIR} pull origin main

%cd {ASSIGNMENT_DIR}

# Cài đặt thư viện cần thiết (giữ nguyên PyTorch CUDA mặc định của Colab)
!pip install -q cityscapesscripts torchmetrics PyYAML tqdm appdirs requests
print("Môi trường và workspace đã sẵn sàng!")

## Bước 3: Cấu Hình Tự Động Push Log & Metrics Về GitHub

In [ ]:
# @title Thiết Lập Git Identity & Hàm Tự Động Đẩy Kết Quả
import os
import subprocess

def auto_push_logs(commit_message="Update training logs and metrics from Colab"):
    """
    Tự động commit và đẩy các file log.csv và metrics_*.json lên GitHub.
    Không commit file trọng số nặng (*.pt) hoặc file dataset.
    """
    if not GITHUB_TOKEN:
        print("[NOTICE] GITHUB_TOKEN trống. Bỏ qua bước auto-push lên GitHub.")
        print("Bạn có thể tải các file log trực tiếp về máy ở bước cuối cùng.")
        return False

    author_name = GIT_USERNAME.strip() or "Colab Contributor"
    author_email = GIT_EMAIL.strip() or "colab@google.com"

    %cd {ASSIGNMENT_DIR}
    !git config user.name "{author_name}"
    !git config user.email "{author_email}"

    # Stage chỉ các file log và metrics
    !git add checkpoints/**/log.csv checkpoints/**/metrics_*.json 2>/dev/null || true

    status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True).stdout
    staged = [line for line in status.splitlines() if line.startswith(('M ', 'A ', 'AM'))]

    if not staged:
        print("[GIT] Không có file log/metrics mới cần commit.")
        return True

    print("[GIT] Các file đang commit:")
    for line in staged:
        print("  ", line)

    !git commit -m "{commit_message}"

    remote_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
    print("[GIT] Đang pull rebase nhánh mới nhất...")
    !git pull --rebase "{remote_url}" {BRANCH}

    print(f"[GIT] Đang push kết quả lên {REPO_OWNER}/{REPO_NAME} ({BRANCH})...")
    res = subprocess.run(["git", "push", remote_url, BRANCH], capture_output=True, text=True)
    if res.returncode == 0:
        print("✅ [GIT] Đã push log và metrics lên GitHub thành công!")
        return True
    else:
        print("❌ [GIT] Lỗi khi push lên GitHub:")
        print(res.stderr)
        return False

if GITHUB_TOKEN:
    print(f"✅ Auto-push đã được kích hoạt cho tài khoản: {GIT_USERNAME} ({GIT_EMAIL})")
else:
    print("ℹ️ Auto-push chưa được bật (không có GITHUB_TOKEN). Bạn sẽ tải log về máy ở Bước 9.")

## Bước 4: Tải Trực Tiếp Dataset Từ Web & Tự Động Giải Nén (Không Cần Drive)

In [ ]:
# @title Tải Trực Tiếp Dataset Từ Web & Giải Nén
import os
import sys
import json
import appdirs
from pathlib import Path
from cityscapesscripts.download import downloader

LOCAL_DATASET = Path(ASSIGNMENT_DIR) / "dataset"
LOCAL_DATASET.mkdir(parents=True, exist_ok=True)

# Kiểm tra nếu dataset đã có sẵn từ trước trong Colab
already_ready = (
    (LOCAL_DATASET / "leftImg8bit_trainvaltest" / "leftImg8bit").is_dir() and
    (LOCAL_DATASET / "gtFine_trainvaltest" / "gtFine").is_dir()
)

if already_ready:
    print("✅ Dataset đã tồn tại đầy đủ trên Colab. Bỏ qua bước tải về!")
else:
    # 1. Tải bằng tài khoản Cityscapes chính thức
    if CITYSCAPES_USERNAME and CITYSCAPES_PASSWORD:
        print(f"🔑 Đang đăng nhập Cityscapes portal với tài khoản: {CITYSCAPES_USERNAME}...")
        cred_dir = appdirs.user_data_dir("cityscapesscripts", "cityscapes")
        os.makedirs(cred_dir, exist_ok=True)
        with open(os.path.join(cred_dir, "credentials.json"), "w", encoding="utf-8") as f:
            json.dump({"username": CITYSCAPES_USERNAME, "password": CITYSCAPES_PASSWORD}, f)

        try:
            session = downloader.login()
            print("✅ Đăng nhập Cityscapes thành công!")
            
            packages_to_download = []
            if not (LOCAL_DATASET / "gtFine_trainvaltest.zip").exists() and not (LOCAL_DATASET / "gtFine_trainvaltest").is_dir():
                packages_to_download.append("gtFine_trainvaltest.zip")
            if not (LOCAL_DATASET / "leftImg8bit_trainvaltest.zip").exists() and not (LOCAL_DATASET / "leftImg8bit_trainvaltest").is_dir():
                packages_to_download.append("leftImg8bit_trainvaltest.zip")

            if packages_to_download:
                print(f"🚀 Đang tải trực tiếp từ web: {packages_to_download} (hỗ trợ resume tải tiếp)...")
                downloader.download_packages(
                    session=session,
                    package_names=packages_to_download,
                    destination_path=str(LOCAL_DATASET),
                    resume=True
                )
                print("✅ Tải hoàn tất cả 2 gói dataset!")
            else:
                print("Các file zip hoặc thư mục đã có sẵn.")
        except Exception as e:
            print(f"❌ Lỗi đăng nhập hoặc tải từ Cityscapes: {e}")
            raise e

    # 2. Hoặc tải từ Custom URL nếu được cung cấp
    elif CUSTOM_GTFINE_URL and CUSTOM_LEFTIMG_URL:
        print("🌐 Đang tải dataset từ Custom Mirror URL...")
        !wget -c "{CUSTOM_GTFINE_URL}" -O "{LOCAL_DATASET}/gtFine_trainvaltest.zip"
        !wget -c "{CUSTOM_LEFTIMG_URL}" -O "{LOCAL_DATASET}/leftImg8bit_trainvaltest.zip"
    else:
        raise ValueError(
            "❌ THIẾU THÔNG TIN: Vui lòng điền CITYSCAPES_USERNAME và CITYSCAPES_PASSWORD "
            "(hoặc link tải CUSTOM_URL) ở BẢNG CẤU HÌNH trên cùng rồi chạy lại!"
        )

    # 3. Giải nén vào thư mục chuẩn hóa
    print("\n📦 Đang giải nén dataset vào SSD tốc độ cao của Colab...")
    
    # Giải nén gtFine
    if not (LOCAL_DATASET / "gtFine_trainvaltest" / "gtFine").is_dir():
        if (LOCAL_DATASET / "gtFine_trainvaltest.zip").is_file():
            print("  • Đang giải nén gtFine_trainvaltest.zip (252 MB)...")
            !unzip -q -o "{LOCAL_DATASET}/gtFine_trainvaltest.zip" -d "{LOCAL_DATASET}/gtFine_trainvaltest"
        elif (LOCAL_DATASET / "gtFine").is_dir():
            (LOCAL_DATASET / "gtFine_trainvaltest").mkdir(parents=True, exist_ok=True)
            !ln -sfn "{LOCAL_DATASET}/gtFine" "{LOCAL_DATASET}/gtFine_trainvaltest/gtFine"

    # Giải nén leftImg8bit
    if not (LOCAL_DATASET / "leftImg8bit_trainvaltest" / "leftImg8bit").is_dir():
        if (LOCAL_DATASET / "leftImg8bit_trainvaltest.zip").is_file():
            print("  • Đang giải nén leftImg8bit_trainvaltest.zip (11.5 GB)...")
            !unzip -q -o "{LOCAL_DATASET}/leftImg8bit_trainvaltest.zip" -d "{LOCAL_DATASET}/leftImg8bit_trainvaltest"
        elif (LOCAL_DATASET / "leftImg8bit").is_dir():
            (LOCAL_DATASET / "leftImg8bit_trainvaltest").mkdir(parents=True, exist_ok=True)
            !ln -sfn "{LOCAL_DATASET}/leftImg8bit" "{LOCAL_DATASET}/leftImg8bit_trainvaltest/leftImg8bit"

    print("✅ Giải nén và chuẩn hóa cấu trúc dữ liệu hoàn tất!")

## Bước 5: Sinh trainIds Masks (Chuyển Đổi 34 Nhãn Gốc $\rightarrow$ 19 Nhãn Huấn Luyện)

In [ ]:
# @title Kiểm Tra và Tạo Nhãn trainIds
import glob

%cd {ASSIGNMENT_DIR}

train_masks = glob.glob(f"{ASSIGNMENT_DIR}/dataset/**/*_gtFine_trainIds.png", recursive=True)

if len(train_masks) == 0:
    print("Đang tạo nhãn trainIds (19 classes + ignore=255) bằng scripts/prepare_data.py...")
    !python scripts/prepare_data.py
    train_masks = glob.glob(f"{ASSIGNMENT_DIR}/dataset/**/*_gtFine_trainIds.png", recursive=True)
    if len(train_masks) == 0:
        raise RuntimeError("Lỗi: Không tìm thấy file trainIds mask sau khi chạy prepare_data.py. Kiểm tra lại thư mục dataset.")
    print(f"✅ Thành công: Đã sinh và xác thực {len(train_masks)} file mask trainIds!")
else:
    print(f"✅ Đã có sẵn {len(train_masks)} file mask trainIds. Sẵn sàng huấn luyện!")

## Bước 6: Nhận Diện GPU & Cập Nhật Tự Động Hyperparameters Vào Configs

In [ ]:
# @title Tự Động Tinh Chỉnh Cấu Hình Theo GPU
import os
import torch
import yaml
from pathlib import Path

cpu_cores = os.cpu_count() or 2

if not torch.cuda.is_available():
    detected_gpu_title = "CPU Fallback (Không có CUDA)"
    auto_batch_size = 4
    auto_workers = 2
    est_per_epoch = "~15 - 20 phút"
    est_total = "~8 - 10 giờ"
else:
    raw_device_name = torch.cuda.get_device_name(0).upper()
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

    if "A100" in raw_device_name:
        detected_gpu_title = f"NVIDIA A100 Tensor Core ({vram_gb:.1f} GB VRAM, HBM2e)"
        auto_batch_size = 16
        auto_workers = min(cpu_cores, 4)
        est_per_epoch = "~40 - 50 giây"
        est_total = "~22 - 25 phút (cả 3 mô hình)"
    elif "L4" in raw_device_name:
        detected_gpu_title = f"NVIDIA L4 Tensor Core ({vram_gb:.1f} GB VRAM, Ada Lovelace)"
        auto_batch_size = 12
        auto_workers = min(cpu_cores, 4)
        est_per_epoch = "~1.2 - 1.4 phút"
        est_total = "~38 - 45 phút (cả 3 mô hình)"
    elif "T4" in raw_device_name:
        detected_gpu_title = f"NVIDIA Tesla T4 ({vram_gb:.1f} GB VRAM, Turing)"
        auto_batch_size = 8
        auto_workers = min(cpu_cores, 2)
        est_per_epoch = "~3.5 - 4.0 phút"
        est_total = "~1.8 - 2.0 giờ (cả 3 mô hình)"
    elif "V100" in raw_device_name:
        detected_gpu_title = f"NVIDIA Tesla V100 ({vram_gb:.1f} GB VRAM, Volta)"
        auto_batch_size = 12
        auto_workers = min(cpu_cores, 4)
        est_per_epoch = "~1.0 - 1.2 phút"
        est_total = "~30 - 35 phút (cả 3 mô hình)"
    else:
        detected_gpu_title = f"{raw_device_name} ({vram_gb:.1f} GB VRAM)"
        auto_batch_size = 16 if vram_gb >= 30 else (12 if vram_gb >= 20 else (8 if vram_gb >= 12 else 4))
        auto_workers = min(cpu_cores, 4)
        est_per_epoch = "Thích ứng"
        est_total = "Thích ứng"

if BATCH_SIZE_MODE == "Auto (Detect GPU)":
    final_batch_size = auto_batch_size
    final_workers = auto_workers
else:
    final_batch_size = CUSTOM_BATCH_SIZE
    final_workers = CUSTOM_NUM_WORKERS

print("=" * 75)
print(f"🤖 PHẦN CỨNG ĐƯỢC NHẬN DIỆN: {detected_gpu_title}")
print(f"   Số CPU Cores khả dụng   : {cpu_cores}")
print("-" * 75)
print("⚙️  HYPERPARAMETERS HOẠT ĐỘNG (Lấy từ Bảng cấu hình trên cùng):")
print(f"   • Batch Size             : {final_batch_size} ({'Tự động tối ưu' if BATCH_SIZE_MODE == 'Auto (Detect GPU)' else 'Tùy chỉnh'})")
print(f"   • Số Dataloader Workers  : {final_workers}")
print(f"   • Tổng số Epochs         : {EPOCHS}")
print(f"   • Head Learning Rate     : {HEAD_LR}")
print(f"   • Partial Backbone LR    : {BACKBONE_LR_PARTIAL}")
print(f"   • Full Backbone LR       : {BACKBONE_LR_FULL}")
print(f"   • Random Seed            : {RANDOM_SEED}")
print(f"   • Mixed Precision (AMP)  : {USE_AMP}")
print(f"   • Giới hạn Batches       : {'Toàn bộ dataset' if MAX_TRAIN_BATCHES == 0 else f'{MAX_TRAIN_BATCHES} batches (Dry-run)'}")
print("-" * 75)
print("⏱️  DỰ TÍNH THỜI GIAN CHẠY:")
print(f"   • Mỗi Epoch              : {est_per_epoch}")
print(f"   • Cả 3 Mô hình           : {est_total}")
print("=" * 75)

# Ghi trực tiếp vào file config yaml trên phiên Colab này
target_configs = [
    ("configs/deeplabv3.yaml", HEAD_LR, None),
    ("configs/resnet_layer3_4_finetune.yaml", HEAD_LR, BACKBONE_LR_PARTIAL),
    ("configs/resnet50_full_deeplabv3.yaml", HEAD_LR, BACKBONE_LR_FULL)
]

for cfg_rel, head_lr_val, backbone_lr_val in target_configs:
    cfg_file = Path(ASSIGNMENT_DIR) / cfg_rel
    if cfg_file.exists():
        with open(cfg_file, "r", encoding="utf-8") as f:
            config_data = yaml.safe_load(f)

        config_data["batch_size"] = final_batch_size
        config_data["num_workers"] = final_workers
        config_data["epochs"] = EPOCHS
        config_data["seed"] = RANDOM_SEED
        config_data["amp"] = USE_AMP
        config_data["pin_memory"] = True
        config_data["eval_every"] = EVAL_EVERY
        config_data["max_train_batches"] = MAX_TRAIN_BATCHES
        config_data["max_eval_batches"] = MAX_EVAL_BATCHES

        config_data["lr"] = head_lr_val
        config_data["head_lr"] = head_lr_val
        if backbone_lr_val is not None:
            config_data["backbone_lr"] = backbone_lr_val

        with open(cfg_file, "w", encoding="utf-8") as f:
            yaml.dump(config_data, f, default_flow_style=False, sort_keys=False)

        print(f"  ✓ Đã cập nhật {cfg_rel}: batch_size={final_batch_size}, epochs={EPOCHS}, lr={head_lr_val}")

print("\n[STATUS] Cấu hình huấn luyện đã sẵn sàng trên phiên Colab này!")

## Bước 7: Thực Hiện Huấn Luyện 3 Mô Hình DeepLabV3

In [ ]:
# @title Model 1: Đóng Băng Backbone (Frozen Backbone) & Auto-Push
# Huấn luyện chỉ ASPP classifier head, đóng băng hoàn toàn ResNet-50 backbone.
import os

%cd {ASSIGNMENT_DIR}
print("🚀 Bắt đầu huấn luyện Model 1: Frozen Backbone...")
!python -m src.train --config configs/deeplabv3.yaml

if os.path.isfile("checkpoints/deeplabv3/local/best.pt"):
    print("✅ Model 1 đã hoàn thành huấn luyện thành công!")
    auto_push_logs("Colab: update Model 1 (Frozen Backbone) training logs")
else:
    print("⚠️ Checkpoint Model 1 chưa được tạo. Kiểm tra log ở trên nếu có lỗi.")

In [ ]:
# @title Model 2: Tinh Chỉnh Một Phần (Partial Fine-Tuning Layer 3 & 4) & Auto-Push
# Đóng băng Layer 1 & 2; tinh chỉnh Layer 3, Layer 4 và Classifier Head.
import os

%cd {ASSIGNMENT_DIR}
print("🚀 Bắt đầu huấn luyện Model 2: Partial Fine-Tuning...")
!python -m src.train --config configs/resnet_layer3_4_finetune.yaml

if os.path.isfile("checkpoints/deeplabv3/resnet_layer3_4_finetune/best.pt"):
    print("✅ Model 2 đã hoàn thành huấn luyện thành công!")
    auto_push_logs("Colab: update Model 2 (Partial Fine-tuning) training logs")
else:
    print("⚠️ Checkpoint Model 2 chưa được tạo. Kiểm tra log ở trên nếu có lỗi.")

In [ ]:
# @title Model 3: Tinh Chỉnh Toàn Bộ (Full Fine-Tuning) & Auto-Push
# Tinh chỉnh toàn bộ backbone ResNet-50 với tốc độ học vi sai (differential lr).
import os

%cd {ASSIGNMENT_DIR}
print("🚀 Bắt đầu huấn luyện Model 3: Full Fine-Tuning...")
!python -m src.train --config configs/resnet50_full_deeplabv3.yaml

if os.path.isfile("checkpoints/deeplabv3/resnet50_full_deeplabv3/best.pt"):
    print("✅ Model 3 đã hoàn thành huấn luyện thành công!")
    auto_push_logs("Colab: update Model 3 (Full Fine-tuning) training logs")
else:
    print("⚠️ Checkpoint Model 3 chưa được tạo. Kiểm tra log ở trên nếu có lỗi.")

## Bước 8: Đánh Giá 3 Mô Hình Trên Tập Test & So Sánh Kết Quả

In [ ]:
# @title Đánh Giá Cả 3 Mô Hình Trên Test Split
import os

%cd {ASSIGNMENT_DIR}

eval_targets = [
    ("Model 1 (Frozen Backbone)", "configs/deeplabv3.yaml", "checkpoints/deeplabv3/local/best.pt", "checkpoints/deeplabv3/local/metrics_test.json"),
    ("Model 2 (Partial Fine-Tuning)", "configs/resnet_layer3_4_finetune.yaml", "checkpoints/deeplabv3/resnet_layer3_4_finetune/best.pt", "checkpoints/deeplabv3/resnet_layer3_4_finetune/metrics_test.json"),
    ("Model 3 (Full Fine-Tuning)", "configs/resnet50_full_deeplabv3.yaml", "checkpoints/deeplabv3/resnet50_full_deeplabv3/best.pt", "checkpoints/deeplabv3/resnet50_full_deeplabv3/metrics_test.json"),
]

for label, cfg_path, ckpt_path, out_json in eval_targets:
    print(f"\n{'='*60}")
    print(f"📊 Đang đánh giá: {label}")
    print(f"{'='*60}")
    if not os.path.isfile(ckpt_path):
        print(f"⚠️ Checkpoint không tồn tại: {ckpt_path}. Bỏ qua đánh giá cho mô hình này.")
        continue
    !python -m src.evaluate --config {cfg_path} \
        --checkpoint {ckpt_path} \
        --split test --max_batches 0 \
        --save_json {out_json}

auto_push_logs("Colab: upload final test evaluation metrics_test.json for all 3 models")

In [ ]:
# @title Bảng Tổng Hợp Kết Quả Đánh Giá
import os
import json
import pandas as pd

models = [
    ("Model 1: Frozen Backbone", "checkpoints/deeplabv3/local/metrics_test.json"),
    ("Model 2: Partial Fine-Tuning", "checkpoints/deeplabv3/resnet_layer3_4_finetune/metrics_test.json"),
    ("Model 3: Full Fine-Tuning", "checkpoints/deeplabv3/resnet50_full_deeplabv3/metrics_test.json"),
]

summary = []
for label, path in models:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        summary.append({
            "Experiment": label,
            "Test mIoU (%)": f"{data.get('miou', 0) * 100:.2f}%",
            "Mean Dice (%)": f"{data.get('mdice', 0) * 100:.2f}%",
            "Pixel Acc (%)": f"{data.get('pixel_acc', 0) * 100:.2f}%",
            "Test Loss": f"{data.get('loss', 0):.4f}",
            "Eval Time (s)": f"{data.get('eval_time_sec', 0):.1f}",
        })

if summary:
    df = pd.DataFrame(summary)
    display(df)
else:
    print("Chưa tìm thấy file metrics_test.json. Hãy chạy bước đánh giá ở trên.")

## Bước 9: Tải Toàn Bộ File Logs & Kết Quả Về Máy

In [ ]:
# @title Tải Các File log.csv & metrics Về Máy Tính Trực Tiếp (Browser Download)
import os
from google.colab import files

export_files = [
    "checkpoints/deeplabv3/local/log.csv",
    "checkpoints/deeplabv3/local/metrics_test.json",
    "checkpoints/deeplabv3/resnet_layer3_4_finetune/log.csv",
    "checkpoints/deeplabv3/resnet_layer3_4_finetune/metrics_test.json",
    "checkpoints/deeplabv3/resnet50_full_deeplabv3/log.csv",
    "checkpoints/deeplabv3/resnet50_full_deeplabv3/metrics_test.json",
]

download_count = 0
for f_path in export_files:
    if os.path.exists(f_path):
        print(f"Đang tải {f_path} về máy...")
        files.download(f_path)
        download_count += 1

if download_count == 0:
    print("Chưa có file log hoặc metrics nào được tạo.")
else:
    print(f"✅ Đã kích hoạt tải {download_count} file kết quả về máy của bạn!")